In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionFusion(nn.Module):
    def __init__(self, text_dim, coord_dim, hidden_dim):
        """
        初始化函数
        :param text_dim: 文字特征向量的维度
        :param coord_dim: 坐标特征向量的维度
        :param hidden_dim: 注意力机制的隐藏维度
        """
        super(AttentionFusion, self).__init__()
        # 将文字特征和坐标特征映射到相同的维度
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.coord_proj = nn.Linear(coord_dim, hidden_dim)
        # 注意力机制中的 Query、Key、Value 线性变换
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        # 缩放因子
        self.scale = hidden_dim ** -0.5

    def forward(self, text_features, coord_features):
        """
        前向传播函数
        :param text_features: 文字特征向量，形状为 (batch_size, text_dim)
        :param coord_features: 坐标特征向量，形状为 (batch_size, coord_dim)
        :return: 融合后的特征向量，形状为 (batch_size, hidden_dim)
        """
        # 将文字特征和坐标特征映射到相同的维度
        text_proj = self.text_proj(text_features)  # (batch_size, hidden_dim)
        coord_proj = self.coord_proj(coord_features)  # (batch_size, hidden_dim)
        
        # 将映射后的特征堆叠起来，形状为 (batch_size, 2, hidden_dim)
        features = torch.stack([text_proj, coord_proj], dim=1)
        
        # 获取 Query、Key 和 Value
        Q = self.query(features)  # (batch_size, 2, hidden_dim)
        K = self.key(features)  # (batch_size, 2, hidden_dim)
        V = self.value(features)  # (batch_size, 2, hidden_dim)
        
        # 计算注意力分数
        attention_scores = torch.matmul(Q, K.transpose(-1, -2)) * self.scale  # (batch_size, 2, 2)
        
        # 应用 Softmax 获取注意力权重
        attention_weights = F.softmax(attention_scores, dim=-1)  # (batch_size, 2, 2)
        
        # 加权求和得到融合后的特征
        fused_features = torch.matmul(attention_weights, V)  # (batch_size, 2, hidden_dim)
        
        # 取融合后的特征向量（可以取加权后的总和或其他操作）
        fused_features = torch.sum(fused_features, dim=1)  # (batch_size, hidden_dim)
        
        return fused_features

# -----------------------------------------------------------------------------示例使用

# 假设参数
text_dim = 300
coord_dim = 4
hidden_dim = 128
batch_size = 10

# 创建模型
model = AttentionFusion(text_dim, coord_dim, hidden_dim)

# 创建示例输入
text_features = torch.randn(batch_size, text_dim)  # (10, 300)
coord_features = torch.randn(batch_size, coord_dim) # (10, 4)

# 前向传播
fused_features = model(text_features, coord_features)
print("Fused features shape:", fused_features.shape)  # 输出融合后特征的形状

Fused features shape: torch.Size([10, 128])
